# finetune 模型 

1. 分类模型-分类目标：判断一个可能的结节是不是真的结节
2. 分类模型-分类目标：判断一个确定的结节是否是恶性肿瘤

# 微调 
1. 微调 本来就是为了节省计算时间和计算资源 同时提高模型的准确率 泛化能力更好一些
2. 用自己的数据 和预训练模型  要求是 自己的数据 和预训练模型 数据 相近 不能是 自己的事动物图片 预训练的事肿瘤图片 这样微调是没有意义的
3. 比如调整最后一层 或者倒数第二层 每增加一层 微调的时间就越长

In [ ]:
# finetune 微调
if self.cli_args.finetune:
    d = torch.load(self.cli_args.finetune, map_location='cpu')

    model_blocks = [n for n, subm in model.named_children() if len(list(subm.parameters()))]

    finetune_blocks = model_blocks[-self.cli_args.finetune_depth:]
    log.info(f'Finetuning {self.cli_args.finetune} blocks: {''.join(finetune_blocks)}')

    model.load_state_dict(
        {k:v for k, v in d['model_state'].items() if k.split('.')[0] not in model_blocks[-1]},
        strict=False,
    )

    for n, p in  model.named_parameters():
        if n.split('.')[0] not in finetune_blocks:
            p.requires_grad (False)

In [1]:
import os
import shutil
import datetime
from util.util import importstr
from util.logconf import logging
log = logging.getLogger('nb')

def run(app, *argv):
    argv = list(argv)
    argv.insert(0, '--num-workers=8')  # <1>
    log.info("Running: {}({!r}).main()".format(app, argv))
    
    app_cls = importstr(*app.rsplit('.', 1))  # <2>
    app_cls(argv).main()
    
    log.info("Finished: {}.{!r}).main()".format(app, argv))

In [ ]:
model_file_path = ''
run('code4.training.ClassificationTrainingApp',f'--epchs=40', '--malignant', '--dateset=MalignantLunaDataSet', '--finetune', model_file_path,'finetune-head')